In [ ]:
# setup (run on every restart)
import os
ds_size = 10000
cache_dir = os.path.join(os.getcwd(), "data", "hf-source")

# Acquiring the Dataset
Using Huggingface's Datasets we can easily load the wikipedia (20231101.en) dataset. Since we are focusing on encyclopedic data, this will serve as the base for testing the dataset generation pipeline.

In [ ]:
from datasets import load_dataset

cache_dir = os.path.join(os.getcwd(), "data", "hf-source")
ds = load_dataset("wikimedia/wikipedia", "20231101.en", cache_dir=cache_dir)

In [ ]:
# verification (optional)
print(f"Dataset length: {len(ds['train'])}")
print(ds['train'][0])
print(ds['train'][-1])

# Pipeline A (UMAP)
Based on [`knoto`](https://github.com/whatphilipcodes/knoto), this pipeline uses `UMAP` to reduce `sBERT` embeddings into two (spatial) dimensions.

In [ ]:
# create subset for dev
ds_subset = ds['train'].select(range(ds_size))
print(f"Test subset length: {len(ds_subset)}")
print(f"Sample entry: {ds_subset[0]['title']}")

In [ ]:
from sentence_transformers import SentenceTransformer
ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", "umap", str(ds_size))

# generate embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [entry['text'] for entry in ds_subset]
print(f"Generating embeddings for {len(texts)} documents...")
embeddings = model.encode(texts, show_progress_bar=True)
print(f"Generated embeddings with shape: {embeddings.shape}")
ds_embed = ds_subset.add_column("embeddings", embeddings.tolist())
print(f"Dataset columns: {ds_embed.column_names}")

# save ds with embeddings
print(f"Saving dataset with embeddings to: {ds_embed_dir}")
ds_embed.save_to_disk(ds_embed_dir)
print("Dataset saved successfully!")

In [ ]:
import umap
import numpy as np
from datasets import load_from_disk

ds_pos_dir = os.path.join(os.getcwd(), "data", "ds_pos", "umap", str(ds_size))

ds_embed = load_from_disk(ds_embed_dir)
print(f"Loaded dataset with {len(ds_embed)} entries")

# collect embeddings
embeddings_array = np.array(ds_embed['embeddings'])
print(f"Embeddings shape: {embeddings_array.shape}")

# apply UMAP reduction
reducer = umap.UMAP(n_components=2, random_state=42)
print("Applying UMAP reduction...")
embed_reduced = reducer.fit_transform(embeddings_array)
print(f"2D embeddings shape: {embed_reduced.shape}")

# add to ds
ds_pos = ds_embed.add_column("x", embed_reduced[:, 0].tolist())
ds_pos = ds_pos.add_column("y", embed_reduced[:, 1].tolist())
print(f"Final dataset columns: {ds_pos.column_names}")

# save ds with positions
ds_pos.save_to_disk(ds_pos_dir)
print("Dataset with UMAP positions saved successfully!")

# Visualization

In [ ]:
from datasets import load_from_disk
import matplotlib.pyplot as plt
import seaborn as sns

ds_pos = load_from_disk(ds_pos_dir)

xs = ds_pos["x"]
ys = ds_pos["y"]

plt.figure(figsize=(6,6))
sns.scatterplot(
    x=xs,
    y=ys,
    s=10,
    alpha=0.7
)
# plt.axis("off")
plt.tight_layout()
plt.show()
